<a href="https://colab.research.google.com/github/dienerlab/2026_strain_course/blob/main/micom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://github.com/dienerlab/2026_strain_course/blob/main/docs/reconstruction/assets/mm_course_2026.png?raw=true)

# 🧫🦠 Part 3: Modeling microbiota-wide metabolism with MICOM

This notebook is the third chapter for the [2026 Course on Strain Metabolic Modeling](https://insights.medunigraz.at/en/events/detail/workshop-strain-level-metabolic-modeling-of-microbial-communities). The presentation slides can be [found here](https://dienerlab.github.io/2026_strain_course/micom).

You can save your own local copy of this notebook by using `File > Save a copy in Drive`. You may be promted to cetify the notebook is safe. We promise that it is 🤞

**Disclaimer:**
The linear and quadratic programming problems MICOM has to solve are very large and very complicated. There are some very good commercial solvers that are very expensive (even though they are often free for academic use). MICOM comes with an included open-source custom-built solver called the "hybrid" solver which we will use here that combines [HIGHS](https://highs.dev) and [OSQP](https://osqp.org/). Solvers usually only guarantee a solution within a certain numerical tolerance of the real solution.

Installing cost-free academic versions of commercial solvers like [IBM CPLEX](https://www.ibm.com/analytics/cplex-optimizer) or [Gurobi](https://www.gurobi.com/) would alow you to reach low tolerances 10<sup>-6</sup> while also being a bit faster.



# 📝 Setup

MICOM installation is is usually pretty straight-forward and can be as easy as typing `pip install micom` into your Terminal.

First let's start by installing our course package again.

In [ ]:
!pip uninstall -y strainy
!pip install -q git+https://github.com/dienerlab/2026_strain_course

## Basic Installation

Installing MICOM is straight-forward in Python. The hybrid solver will be installed along with it by default.

---

**💡: Info on the MICOM version**

Here in the course we will install a beta versin of the newest MICOM (1.0) which comes with a new interface, called the "Batch" interfacce. Because this will be the default interface for all coming MICOM versions we will learn it now. Normally, you dont' have to specify a MICOM version and can just use `pip install micom`, but here we will specify it so we get the beta and and not an older version.

---

In [ ]:
!pip install -q micom==1.0.0b4

Okay, all done. So let's get started building some models 🦺🛠😁.

# 💻 MICOM

We will use the Python interface to MICOM since it plays nicely with Colaboratory. However, you could run the same steps within the QIIME 2 MICOM plugin ([q2-micom](https://library.qiime2.org/plugins/q2-micom/26/)).

Here is an overview of all the steps and functions across both interfaces:
![micom overview](https://github.com/micom-dev/q2-micom/raw/706f583a060b91c12c0cec7acea2354fdd0dd320/docs/assets/overview.png)

New version of MICOM manage your experimental setup in a unit called `Batch`, which consists of one or several microbial communities, each possibly containing many taxa. So it can represent a single community or culture (think of "batch culture") or many of those microbial communities (think of a batch of samples or experiments). To set up a MICOM Batch we need a few things:

1. An abundance table that specifies how much of each taxon is in each sample.
2. [optional] A model database that contains metabolic reconstructions for each of the taxa in the abundance table.
3. [optional] A growth medium/environment denoting the flux bounds coming from the environment.

So the only *required* information is a combined abundance table that also includes the taxonopmy for each taxon, referred to hereafter as a taxonomy table. Let's load a sample taxonomy table to see what it looks like:



In [ ]:
from micom.data import test_data

test_data()

In this taxonomy table, we see four identical strains of _E. coli_ (a through d), across two samples (sample_1 and sample_2). We can see that each row represents a single taxon in a single sample, and the `abundance` column identifies the abundance of that taxon in the sample.

The `id` column here is optional and can be used to give custom names to taxa.  It should be expressive and not include spaces or special characters. Note that this name should be unique to the taxon, *not* the sample. So strain a should be called "strain_a" in all samples it appears in.

Okay, but what is the abundance table from our data set? Let's get it and take a look:

In [ ]:
from strainy import gimme_abundances

ab = gimme_abundances()
ab

In [ ]:
ab.columns

This does look a bit like the other table but contains much more information. Basically, MICOM only checks for the required columns but does allow you to include arbitrary additional ones if you like. This is often useful to add in sample or taxon metadata. Here we added in a bunch of metadata for our samples. It's a bit hard to see this in the full table so let's condense this a bit:

In [ ]:
ab[["sample_id", "host_age_days", "host_del_route", "host_sex", "host_birthweight_g", "host_diet"]].drop_duplicates()

Okay, so we have infos about the infants and their delivery mode. The abundance here are the reads assigned to each strain. Cool, that seems to make sense.

## Model databases

Basically, a specific model database can be used to quickly summarize pangenome-scale models for use within MICOM. So, before we read our data we have to decide which model database to use. We will go with the [AGORA database](https://pubmed.ncbi.nlm.nih.gov/27893703/), which is a curated database of more than 800 bacterial strains that commonly live in the human gut. In particular, we will use a version of this database summarized on the genus rank which can be downloaded from the [MICOM data repository](https://doi.org/10.5281/zenodo.3755182), which contains a whole lot of prebuilt databases. This database is available from the materials folder that we previously cloned.

In [ ]:
from strainy import gimme_strain_db

man = gimme_strain_db()
man

## The growth medium/environment

The last thing we can provide is also the easiest, because we have already dealt with it in the first parts of the course: the growth medium. The format for growth media in MICOM is pretty similar to what we saw in GAPSEQ. One important thing to note is that growth media are specific to the universal model, and thus, the reconstruction method used. So a GAPSEQ medium will work fine for the GAPSEQ model database here, but won't match models from the VMH/AGORA universe.

In [ ]:
from strainy import gimme_medium

medium = gimme_medium(format = "micom")
medium

With our taxonomy table ready to go, and our metadata merged, its finally time to get to the modeling part! 🎉

## The Batch

Like we mentioned MICOM uses the concept of a `Batch` which we can now start.

In [ ]:
from micom.batch import Batch

infants = Batch(taxonomy=ab, medium=medium, model_db="infants_gtdb232_strain_1.zip")
infants

This just sets up the initial data structure. We see that we have the 8 samples in here and that there are 51 taxa across all samples. We also get some info about our medium and so on.

One thing that is important is that each `Batch` comes with a configuration that defines a lot of settings for the simulations.

In [ ]:
infants.config

This chooses some sensible defaults but can be adjusted as you see fit. For instance we see that it defaults to 2 threads, but our Colab environment only has 2 CPUs so we will set this to 2. Because we want to log what we have changed let us also save this configuration.

In [ ]:
infants.config.threads = 2

infants.config.to_yaml("my_config.yml")

infants.config

We went through some of those settings in the course but you can always find an explanation of all the settings in the [documentation]().

## Building models

As the first step we need to build the microbial community metabolic models (MCMMs) for each sample. This will combine the individual strain models as discussed in the course into a larger community model and set up the correct abundance scaling and exchange reactions with the external environment.

This is really quite straight-forward and the only thing we need to specify is the location of our community models. Let's get started. This should take around 15 minutes.

In [ ]:
infants.build("models")

This will tell you how many taxa were found in the database and what fraction of the total abundance was represented by the database. This build manifest is also saved in the `infants.build_manifest` slot.

In [ ]:
infants.build_manifest.head()

A very basis summary is also provided in the Batch overview.

In [ ]:
infants

### Growing the models
Great, now we have our media & our models, it's time to get growing. This will take some time, so we'll use that time as an opportunity to discuss more in depth what these processes do, and what to look for in the results. First, let's run the `grow()` command. This will take the models we've built, and find an optimal solution to the fluxes based upon the medium that's been applied. Should take about 10 minutes to complete.

In [ ]:
results = infants.grow()

After this has run another box is ticked in our Batch.

In [ ]:
infants

Again, if that takes too long or was aborted, we can read it in from the treasure chest.

In [ ]:
from micom.batch import load_results
from strainy import gimme_treasure

try:               # Will only run if the previous step failed
  results
except NameError:
  gimme_treasure("growth.zip")
  results = load_results("growth.zip")

What kind of results did we get? Well, [grow]() returns a [GrowthResults](https://micom-dev.github.io/micom/autoapi/micom/workflows/results/index.html#micom.workflows.results.GrowthResults) object that contains three things:

1. **growth_rates** - the predicted growth rate for all taxa in all samples
2. **exchanges** - the import and export fluxes for each taxon and the external environment
3. **annotations** - annotations for the fluxes mapping to other databases

### 📈 Growth Rates

The growth rates are pretty straightforward.

In [ ]:
results.growth_rates

### ↔️ Exchange Fluxes

More interesting are the exchange fluxes. These reactions represent the import and export of metabolites from the system Let's look at those now:

In [ ]:
results.exchanges

So we see how much of each metabolite is either consumed or produced by each taxon in each sample. `tolerance` denotes the accuracy of the solver and tells you the smallest absolute flux that is likely different form zero (i.e., substantial flux). *All of the fluxes are normalized to 1g dry weight of bacteria*. So, you can directly compare fluxes between taxa, even if they are present at very different abundances.

If you're curious what the abbreviation for each of these metabolites represents, that can be found in the annotations dataframe. For instance, let's find out what `"cpd00066_e0"` represents.

In [ ]:
anns = results.annotations
anns[anns.metabolite == "cpd00066_e0"]

Phenylalanine! All of these annotations and more information at are also available at the [ModelSEED website](https://modelseed.org).


# 📊 Visualizations

Let's visualize our results. Because of the rich output of these models, it can be overwhelming to represent it all, but don't worry! There are tools in place for this already.

We will use the standard visualizations included in MICOM. These tools take in the growth results we obtained before and create visualizations in standalone HTML files that bundle the plots and raw data and can be viewed directly in your browser.

First, let's look at the growth rates of each taxon across samples.

In [ ]:
from micom.viz import *

viz = plot_growth(results)

Normally, we could call `viz.view()` afterwards and it would open it in our web browser. However, this will not work in Colab. However, the plot function creates the file `growth_rates_[DATE].html` in your `materials` folder. To open it, simply download that file and view it in your web browser. We can see that there are many things going on, but it's not super clear. Let's continue.

## Growth niches

Another thing we can look at is whether individual taxa inhabit different growth niches across different disease contexts. Here we can use the `plot_exchanges_per_taxon` function to see how exchanges differ within and between taxa, within and across different infant microbiota.

In [ ]:
plot_exchanges_per_taxon(results, direction="import")



This function projects the full set of import or export fluxes onto a two dimensional plane, and arranges taxa so that more similar flux patterns lie nearer together. Taxa closer to one another compete for a more similar set of resources (and/or produce a more similar set of metabolites). The center of the plot signifies a more competitive nutrient space, whereas clusters on the outskirts denote more isolated niches.

You can tune [TSNE parameters](https://distill.pub/2016/misread-tsne/), such as perplexity, to get a more meaningful grouping. We will lower the perplexity here since we don't have a lot of data points.


## Associations with a phenotype

We can also ask the question whether particular exchange flux patterns are associated with a particular phenotype. Here, we will check whether we get an association with the delivery mode of the infants.

First we start by preparing a mapping between sample id and the delivery route.

In [ ]:
delivery_mode = infants.build_manifest.host_del_route
delivery_mode.index = infants.build_manifest.sample_id
delivery_mode

Now we can run the association analysis. By default this will run Mann-Whitney tests for all production fluxes in the data set for the samples where the metabolite was produced. We only have 4 samples per group and most metabolites will not be produced by the microbiome in all of them, so we probably won't get any significant hits after correcting for multiple testing. To still see some trends we will show all of the results.

In [ ]:
r = plot_association(results, delivery_mode, fdr_threshold=1.0)

In [ ]:
r.data["tests"].sort_values(by="q")

Nothing significant but some interesting metabolites associated with the delivery. Delivery mode can also be predicted quite well just from the flux patterns it seems.

## Metabolic Interaction analyses

Now where we have some global overview we could ask ourselves what happens in detail. One aspect we have not looked at is the interactions between different strains which are also predicted by MICOM.

One first question we might have is which metabolites are exchanges particularly often in the communities and whether those patterns of metabolic exchanges are affected by the phenotype. For this we can use the metabolic exchanges score (MES). The MES is the geometric means of producers and consumers for each metabolite

$$
MES = 2\cdot\frac{P\cdot C}{P+C}
$$

It goes to zero if a metebolite is not exchanges at all and rises the more producer-consumer pairs there are in a sample.

We will quantify this here and also add in our delivery route again.

In [ ]:
plot_mes(results, groups=delivery_mode)

If we really want to go down the rabbit hole we could now ask which metabolites a strain of interest (focal strain) is exchanging with which other taxa in the data set.

Let's do this for the strain we originall reconstructed. You might have to go back and find the original strain identifier. It should start with `t__`.

In [ ]:
plot_focal_interactions(results, taxon="t__[YOUR STRAIN HERE]")

# 🏫 Exercises

**1. Functional variability**

Are there actually large differences in the production of metabolites? Well, *you"* tell us. Use the [`plot_exchanges_per_sample`](https://micom-dev.github.io/micom/autoapi/micom/viz/exchanges/index.html#micom.viz.exchanges.plot_exchanges_per_sample) function to visualize the *production* rates.

In [ ]:
# Your code here

**2. Bifidobacteria impact**

Okay this might be bit more difficult because you need some familiarity with [pandas](https://pandas.pydata.org/) (or AI 😏)...

So we know that Bifidobacteria are pretty important. Is that also true for the small data set here? To check create a new "phenotype" series that assigns each sample the "positive" or "negative" status based on whether it contains a strain from the Bifidobacterium genus. Then run `plot_association` with this new series again and check the results.

In [ ]:
# Your code here

# 🔵 Addendum


## Choosing a tradeoff value

Even if you don't have growth rates available you can still use your data to choose a decent tradeoff value. This can be done by choosing the largest tradeoff value that still allows growth for the majority of the taxa that you observed in the sample (if they are present at an appreciable abundance, they should be able to grow). This can be done with the `tradeoff` workflow in MICOM that will run cooperative tradeoff with varying tradeoff values, which can be visualized with the `plot_tradeoff` function.

Note that this will take quite long here (about 3h). So for now let us read the results from the treasure chest and check.

In [ ]:
from strainy import gimme_treasure

gimme_treasure("tradeoffs.csv")

Now we can plot them.

In [ ]:
import pandas as pd
from micom.viz import *

trs = pd.read_csv("tradeoffs.csv")
plot_tradeoff(trs)

To actually run the anlysis it is again a batch command.

In [ ]:
from micom.viz import *

trs = infants.tradeoff()

plot_tradeoff(trs)

After opeing `tradeoff_[DATE].html` you will see that, for our example here, most tradeoff values work great.

![tradeoffs](https://dienerlab.github.io/2026_strain_course/micom/assets/tradeoffs.png)

This is because we modeled very few taxa, which keeps the compettion down. If you would allow for fewer abundant taxa in the models, this would change drastically.


For instance, here is an example from a colorectal cancer data set:

[![tradeoff example](https://micom-dev.github.io/micom/_images/tradeoff.png)](https://micom-dev.github.io/micom/_static/tradeoff.html)

You can see how not using the cooperative tradeoff would give you nonsense results where only 10% of all observed taxa grew. A tradeoff value of 0.6-0.8 would probably be a good choice for this particular data set.